# Environment, Dependencies & Configuration Audit

In [1]:

# Cell 1: Environment, Dependencies & Foundational Configuration Audit

import sys
import platform
import importlib
from pathlib import Path

import pandas as pd
from rich.console import Console
from rich.table import Table

console = Console()

# -------------------------------------------------------------------------
# 1. Detect notebook location
# -------------------------------------------------------------------------
current_dir = Path.cwd().resolve()

# Assumes the notebook is either in PROJECT_ROOT/notebooks/
# or launched with the working directory set to PROJECT_ROOT.
notebook_detected_root = (
    current_dir.parent
    if current_dir.name == "notebooks"
    else current_dir
)

if str(notebook_detected_root) not in sys.path:
    sys.path.insert(0, str(notebook_detected_root))

# -------------------------------------------------------------------------
# 2. Python environment
# -------------------------------------------------------------------------
python_version = platform.python_version()
platform_info = platform.platform()

assert sys.version_info >= (3, 10), (
    f"Python >= 3.10 required. Detected: {python_version}"
)

# -------------------------------------------------------------------------
# 3. Dependency audit
# -------------------------------------------------------------------------
REQUIRED_LIBS = [
    "pydantic",
    "pydantic_settings",
    "pypdf",
    "docx",
    "sentence_transformers",
    "faiss",
    "dotenv",
    "langchain_text_splitters",
]

dep_audit = []

for lib in REQUIRED_LIBS:
    try:
        mod = importlib.import_module(lib)
        version = getattr(mod, "__version__", "Installed; version unknown")

        # faiss often exposes its version differently.
        if lib == "faiss" and version == "Installed; version unknown":
            version = getattr(mod, "__version__", version)

        dep_audit.append({
            "Library": lib,
            "Status": "PASS",
            "Version": str(version),
        })

    except ImportError as exc:
        dep_audit.append({
            "Library": lib,
            "Status": "FAIL",
            "Version": str(exc),
        })

display(pd.DataFrame(dep_audit))

# -------------------------------------------------------------------------
# 4. Load canonical settings
# -------------------------------------------------------------------------
try:
    from src.config.settings import settings
    from src.config.model_config import (
        model_settings,
        chunking_config,
        retrieval_config,
    )

    # Use the settings module as the source of truth.
    configured_root = settings.PROJECT_ROOT.resolve()

    # Normalize paths for accurate comparisons.
    raw_dir = settings.DATA_RAW_DIR.resolve()
    processed_dir = settings.DATA_PROCESSED_DIR.resolve()
    artifacts_dir = settings.ARTIFACTS_DIR.resolve()
    logs_dir = settings.LOGS_DIR.resolve()

    settings_status = "Loaded successfully"
    settings_error = None

    api_provider = model_settings.API_PROVIDER
    embedding_dim = model_settings.EMBEDDING_DIMENSION
    api_based = model_settings.API_BASED

except Exception as exc:
    settings_status = "FAILED"
    settings_error = repr(exc)

    configured_root = None
    raw_dir = None
    processed_dir = None
    artifacts_dir = None
    logs_dir = None

    api_provider = "N/A"
    embedding_dim = "N/A"
    api_based = "N/A"

# -------------------------------------------------------------------------
# 5. Compare notebook root against settings root
# -------------------------------------------------------------------------
rows = [
    ("Current Working Directory", str(current_dir)),
    ("Notebook-Detected Root", str(notebook_detected_root)),
    ("Settings Status", settings_status),
]

if configured_root is not None:
    rows.extend([
        ("Settings Project Root", str(configured_root)),
        (
            "Notebook Root Matches Settings Root",
            str(notebook_detected_root.resolve() == configured_root),
        ),
        ("Project Root Exists", str(configured_root.is_dir())),
        ("data/ Exists", str((configured_root / "data").is_dir())),
        ("data/raw Exists", str(raw_dir.is_dir())),
        ("data/processed Exists", str(processed_dir.is_dir())),
        ("artifacts/ Exists", str(artifacts_dir.is_dir())),
        ("logs/ Exists", str(logs_dir.is_dir()),
        ),
        ("Raw Data Path", str(raw_dir)),
        ("Processed Data Path", str(processed_dir)),
        ("Artifacts Path", str(artifacts_dir)),
        ("Logs Path", str(logs_dir)),
        ("API-Based Mode", str(api_based)),
        ("LLM Provider", str(api_provider)),
        ("Embedding Dimension", str(embedding_dim)),
    ])

else:
    rows.append(("Settings Error", str(settings_error)))

table = Table(title="Part 1: Environment & Configuration Audit")
table.add_column("Parameter", style="cyan")
table.add_column("Value", style="green")

for parameter, value in rows:
    table.add_row(parameter, value)

console.print(table)

# -------------------------------------------------------------------------
# 6. Explicit diagnostic
# -------------------------------------------------------------------------
if configured_root is not None:
    if notebook_detected_root.resolve() != configured_root:
        console.print(
            "\n[bold red]ROOT MISMATCH[/bold red]\n"
            "The notebook-detected root and settings root differ. "
            "Use settings.PROJECT_ROOT as the canonical root."
        )
    else:
        console.print(
            "\n[bold green]ROOT MATCH[/bold green]\n"
            "The notebook and settings resolve to the same project root."
        )

    missing_dirs = [
        path
        for path in [raw_dir, processed_dir, artifacts_dir]
        if not path.is_dir()
    ]

    if missing_dirs:
        console.print("\n[yellow]Directories that do not exist:[/yellow]")
        for path in missing_dirs:
            console.print(f"  - {path}")
        console.print(
            "This audit did not create them. Create them in the "
            "appropriate initialization or indexing workflow."
        )
    else:
        console.print(
            "[bold green]All checked data/artifact directories exist.[/bold green]"
        )


/home/ahmed/anaconda3/envs/genai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Library,Status,Version
0,pydantic,PASS,2.14.1
1,pydantic_settings,PASS,2.16.1
2,pypdf,PASS,6.21.1
3,docx,PASS,1.2.0
4,sentence_transformers,PASS,6.1.0
5,faiss,PASS,1.15.1
6,dotenv,PASS,Installed; version unknown
7,langchain_text_splitters,PASS,Installed; version unknown


                              Part 1: Environment & Configuration Audit                               
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Parameter                           ┃ Value                                                        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Current Working Directory           │ /run/media/ahmed/New Volume/RAG/HR-Policy-RAG/notebooks      │
│ Notebook-Detected Root              │ /run/media/ahmed/New Volume/RAG/HR-Policy-RAG                │
│ Settings Status                     │ Loaded successfully                                          │
│ Settings Project Root               │ /run/media/ahmed/New Volume/RAG/HR-Policy-RAG                │
│ Notebook Root Matches Settings Root │ True                                                         │
│ Project Root Exists                 │ True                                                         │
│ data/ Exists                        │ True                                                         │
│ data/raw Exists                     │ True                                                         │
│ data/processed Exists               │ True                                                         │
│ artifacts/ Exists                   │ True                                                         │
│ logs/ Exists                        │ True                                                         │
│ Raw Data Path                       │ /run/media/ahmed/New Volume/RAG/HR-Policy-RAG/data/raw       │
│ Processed Data Path                 │ /run/media/ahmed/New Volume/RAG/HR-Policy-RAG/data/processed │
│ Artifacts Path                      │ /run/media/ahmed/New Volume/RAG/HR-Policy-RAG/artifacts      │
│ Logs Path                           │ /run/media/ahmed/New Volume/RAG/HR-Policy-RAG/logs           │
│ API-Based Mode                      │ True                                                         │
│ LLM Provider                        │ groq                                                         │
│ Embedding Dimension                 │ 384                                                          │
└─────────────────────────────────────┴──────────────────────────────────────────────────────────────┘

ROOT MATCH
The notebook and settings resolve to the same project root.

All checked data/artifact directories exist.

In [2]:
# Cell 2: Test File and Text Utilities against edge cases
import tempfile
from src.utils.file_utils import (
    ensure_directory,
    compute_file_hash,
    save_json,
    load_json,
    list_files_by_extension
)
from src.utils.text_utils import (
    normalize_whitespace,
    clean_markdown_artifacts,
    estimate_token_count
)

test_results = []

def run_test(name, func, expected_check, *args, **kwargs):
    try:
        result = func(*args, **kwargs)
        passed = expected_check(result)
        test_results.append({"Test": name, "Status": "PASS" if passed else "FAIL", "Details": str(result)[:60]})
    except Exception as ex:
        test_results.append({"Test": name, "Status": "EXCEPTION", "Details": str(ex)})

# --- Text Utils Testing ---
# 1. Empty & malformed inputs
run_test("text: empty string", normalize_whitespace, lambda r: r == "", "")
run_test("text: none input", normalize_whitespace, lambda r: r == "", None)
run_test("text: repeated whitespace/newlines", normalize_whitespace, lambda r: r == "line1\n\nline2", "   line1 \t\t \n\n\n\n line2   ")

# 2. Unicode ligatures (fi, fl) normalization
run_test("text: ligatures NFKC", normalize_whitespace, lambda r: r == "offline flight", "of\ufb01ne \ufb02ight")

# 3. Arabic & English mixed text
arabic_sample = "سياسة الإجازات   السنوية  HR Policy: 2026\n\n\nتطبق   على الجميع"
run_test("text: Arabic-English mixed", normalize_whitespace, lambda r: "سياسة الإجازات السنوية HR Policy: 2026\n\nتطبق على الجميع" in r, arabic_sample)

# 4. Control characters & Markdown artifacts
dirty_str = "# Title\x00\x1f\nText with null and [source](link)"
run_test("text: clean control chars", clean_markdown_artifacts, lambda r: "\x00" not in r and "\x1f" not in r, dirty_str)

# 5. Token count estimation
run_test("text: token estimation normal", estimate_token_count, lambda r: r > 0, "Enterprise policy statement")
run_test("text: token estimation empty", estimate_token_count, lambda r: r == 0, "")

# --- File Utils Testing ---
with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)

    # 6. Ensure directory creation
    sub_dir = tmp_path / "deep" / "nested" / "dir"
    run_test("file: ensure directory", ensure_directory, lambda p: p.exists() and p.is_dir(), sub_dir)

    # 7. Empty file hashing
    empty_file = tmp_path / "empty.txt"
    empty_file.touch()
    run_test("file: empty file SHA256", compute_file_hash, lambda h: len(h) == 64, empty_file)

    # 8. Unicode & Arabic JSON save/load
    json_path = tmp_path / "test.json"
    data = {"company": "الشركة العالمية", "policies": ["إجازات", "رواتب"], "active": True}
    save_json(data, json_path)
    run_test("file: json save & load roundtrip", load_json, lambda d: d == data, json_path)

    # 9. Invalid paths
    non_existent = tmp_path / "missing.json"
    try:
        load_json(non_existent)
        test_results.append({"Test": "file: load non-existent path", "Status": "FAIL", "Details": "Did not raise FileNotFoundError"})
    except FileNotFoundError:
        test_results.append({"Test": "file: load non-existent path", "Status": "PASS", "Details": "Correctly raised FileNotFoundError"})

    # 10. Listing by extensions
    f_pdf = tmp_path / "doc1.pdf"
    f_docx = tmp_path / "doc2.DOCX"
    f_pdf.touch()
    f_docx.touch()
    run_test("file: list extensions case-insensitive", list_files_by_extension, lambda fl: len(fl) == 2, tmp_path, ["pdf", ".docx"])

display(pd.DataFrame(test_results))

,Test,Status,Details
0,text: empty string,PASS,
1,text: none input,PASS,
2,text: repeated whitespace/newlines,PASS,line1\n\nline2
3,text: ligatures NFKC,FAIL,offine flight
4,text: Arabic-English mixed,PASS,سياسة الإجازات السنوية HR Policy: 2026\n\nتطبق...
5,text: clean control chars,PASS,# Title\nText with null and [source](link)
6,text: token estimation normal,PASS,6
7,text: token estimation empty,PASS,0
8,file: ensure directory,PASS,/tmp/tmpb718jtce/deep/nested/dir
9,file: empty file SHA256,PASS,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...


In [3]:
# Cell 3: Scan data/raw for HR policy documents & verify against DocumentRegistry
from pathlib import Path
import pandas as pd
from src.config.settings import settings
from src.utils.file_utils import list_files_by_extension, ensure_directory
from src.ingestion.document_registry import DocumentRegistry

# Dynamically ensure raw_dir points to project_root / data / raw
current_dir = Path.cwd()
root_dir = current_dir.parent if current_dir.name == "notebooks" else current_dir
raw_dir = root_dir / "data" / "raw"
ensure_directory(raw_dir)

# Discover files (case-insensitive for PDF and DOCX)
discovered_pdfs = list_files_by_extension(raw_dir, ["pdf"])
discovered_docxs = list_files_by_extension(raw_dir, ["docx"])
all_discovered = discovered_pdfs + discovered_docxs

# Verify against registry
registry_path = root_dir / "artifacts" / "document_registry.json"
ensure_directory(registry_path.parent)
registry = DocumentRegistry(registry_path)

corpus_summary = []
for p in all_discovered:
    corpus_summary.append({
        "File Name": p.name,
        "Extension": p.suffix.lower(),
        "Size (KB)": round(p.stat().st_size / 1024, 2),
        "Is Registered": registry.is_document_registered(p),
        "Absolute Path": str(p)
    })

corpus_df = pd.DataFrame(corpus_summary)
print(f"Total Raw Documents Found: {len(corpus_df)} (PDF: {len(discovered_pdfs)}, DOCX: {len(discovered_docxs)})")
display(corpus_df)

Total Raw Documents Found: 3 (PDF: 3, DOCX: 0)


,File Name,Extension,Size (KB),Is Registered,Absolute Path
0,employee-handbook-and-policies-manual.pdf,.pdf,65.29,False,/run/media/ahmed/New Volume/RAG/HR-Policy-RAG/...
1,HR Policy.pdf,.pdf,173.48,False,/run/media/ahmed/New Volume/RAG/HR-Policy-RAG/...
2,Introduction-to-Human-Resource-Management-1748...,.pdf,16637.46,False,/run/media/ahmed/New Volume/RAG/HR-Policy-RAG/...


In [4]:
# Cell 4: Parse documents, test selectable text vs scanned pages, and evaluate OCR requirements
from src.ingestion.loader_factory import LoaderFactory
from src.preprocessing.cleaner import DocumentCleaner
from src.preprocessing.metadata_extractor import MetadataExtractor

audit_records = []

for file_path in all_discovered:
    record = {
        "File Name": file_path.name,
        "Type": file_path.suffix.lower().lstrip("."),
        "Total Pages": 0,
        "Raw Chars": 0,
        "Cleaned Chars": 0,
        "Avg Chars/Page": 0,
        "Selectable Text %": 0.0,
        "Requires OCR": False,
        "Detected Title": None,
        "Parsing Error": None
    }

    try:
        # 1. Parse via LoaderFactory
        doc = LoaderFactory.load_document(file_path)
        record["Total Pages"] = doc.metadata.total_pages or len(doc.pages)
        record["Raw Chars"] = len(doc.content)

        # 2. Page-level extraction analysis (Detect scanned / non-selectable pages)
        selectable_pages = 0
        total_pages = max(1, len(doc.pages))

        for page in doc.pages:
            # A page with fewer than 30 characters is likely scanned, blank, or an image
            if len(page.text.strip()) >= 30:
                selectable_pages += 1

        record["Selectable Text %"] = round((selectable_pages / total_pages) * 100, 1)

        # If more than 20% of pages are empty while total file size is non-trivial, OCR might be needed
        if record["Selectable Text %"] < 80.0 and file_path.stat().st_size > 15_000:
            record["Requires OCR"] = True

        # 3. Clean and extract metadata
        cleaned_doc = DocumentCleaner.clean_document(doc)
        enriched_doc = MetadataExtractor.enrich_document_metadata(cleaned_doc)

        record["Cleaned Chars"] = len(enriched_doc.content)
        record["Avg Chars/Page"] = round(record["Cleaned Chars"] / total_pages, 1)
        record["Detected Title"] = enriched_doc.metadata.title

    except Exception as e:
        record["Parsing Error"] = str(e)
        record["Requires OCR"] = True

    audit_records.append(record)

audit_df = pd.DataFrame(audit_records)
display(audit_df)

,File Name,Type,Total Pages,Raw Chars,Cleaned Chars,Avg Chars/Page,Selectable Text %,Requires OCR,Detected Title,Parsing Error
0,employee-handbook-and-policies-manual.pdf,pdf,10,6390,6390,639.0,100.0,False,Employee Handbook and Policies Manual | Policies,None
1,HR Policy.pdf,pdf,10,18710,18710,1871.0,100.0,False,Hr Policy,None
2,Introduction-to-Human-Resource-Management-1748...,pdf,527,883965,883965,1680.5,97.3,False,Introduction to Human Resource Management,None


In [5]:
# Cell 5: Synthesis, Metrics Logging, and Ingestion Architecture Decision
ocr_needed_count = audit_df["Requires OCR"].sum() if not audit_df.empty else 0
parsing_failures = audit_df[audit_df["Parsing Error"].notnull()] if not audit_df.empty else []
total_docs = len(audit_df)

console.print("\n[bold green]================ EXPERIMENT AUDIT FINDINGS ================[/bold green]")
print(f"Total Documents Audited: {total_docs}")
print(f"Parsing Failures: {len(parsing_failures)}")
print(f"Documents Flagged for OCR: {ocr_needed_count}")

# Print Architectural Decision
decision_title = "[bold yellow]ARCHITECTURAL DECISION: LOADERS & PREPROCESSING[/bold yellow]"
console.print(f"\n{decision_title}")

if ocr_needed_count == 0 and len(parsing_failures) == 0 and total_docs > 0:
    verdict = """
    [bold green]DECISION: Standard Text Loaders (No OCR Required)[/bold green]
    - Status: All analyzed PDF and DOCX files contain digital, selectable text streams.
    - Loaders: Retain pypdf.PdfReader (src/ingestion/pdf_loader.py) and python-docx (src/ingestion/docx_loader.py).
    - Recommendation: DO NOT integrate heavy OCR engines (e.g., Tesseract, EasyOCR, PyMuPDF-OCR) or system binaries into deployment.
    - Preprocessing: Standardize on unicodedata.normalize('NFKC'), regex whitespace normalization, and DocumentCleaner boilerplate removal.
    """
elif total_docs == 0:
    verdict = """
    [bold cyan]DECISION: Corpus Directory Empty[/bold cyan]
    - Status: data/raw/ contains 0 files.
    - Action: Place sample HR policy PDF and DOCX documents in data/raw/ and re-run this notebook before indexing.
    - Default Architecture: Proceed with lightweight PDFLoader & DocxLoader without OCR until scanned assets are confirmed.
    """
else:
    verdict = f"""
    [bold red]DECISION: OCR Fallback Pipeline Required[/bold red]
    - Status: {ocr_needed_count} document(s) have unextractable text or parsing errors.
    - Action: Introduce pdf2image + pytesseract or rapidocr-onnx into src/ingestion/pdf_loader.py as a fallback loader.
    """

console.print(verdict)

================ EXPERIMENT AUDIT FINDINGS ================

Total Documents Audited: 3
Parsing Failures: 0
Documents Flagged for OCR: 0


ARCHITECTURAL DECISION: LOADERS & PREPROCESSING

DECISION: Standard Text Loaders (No OCR Required)
    - Status: All analyzed PDF and DOCX files contain digital, selectable text streams.
    - Loaders: Retain pypdf.PdfReader (src/ingestion/pdf_loader.py) and python-docx (src/ingestion/docx_loader.py).
    - Recommendation: DO NOT integrate heavy OCR engines (e.g., Tesseract, EasyOCR, PyMuPDF-OCR) or system binaries
into deployment.
    - Preprocessing: Standardize on unicodedata.normalize('NFKC'), regex whitespace normalization, and 
DocumentCleaner boilerplate removal.